# Download rápido de um frame AIA (SDO)

Notebook simples para baixar **um único frame** do AIA/SDO num instante específico. Útil para:
- conferência visual rápida de um instante do Sol antes de decidir uma janela de download maior;
- testar um comprimento de onda ou data diferente sem rodar o pipeline completo;
- gerar uma imagem isolada para ilustração/apresentação.

**Quando usar os outros notebooks desta pasta em vez deste:**
- `download-sdo-lightcurves.ipynb` — baixa o intervalo completo de frames (múltiplos, a cada 10 min), para qualquer comprimento de onda do AIA, nos 4 eventos de referência do projeto. Necessário sempre que for rodar a simulação de trânsito.
- `main-fft-analysis.ipynb` — roda a simulação de trânsito e a análise de resíduo/FFT/PCA sobre os dados baixados pelo notebook acima.

**Diferença importante:** este notebook baixa o arquivo para o diretório atual (uso exploratório/descartável), **não** para `Sun/sdo_aia_download/<comprimento_de_onda>/...` — a estrutura de pastas usada pelo restante do projeto e esperada por `Estrela(useFits=True, fits_path=...)`. Se o objetivo for alimentar o pipeline de simulação, use `download-sdo-lightcurves.ipynb`.

## Imports

In [ ]:
import matplotlib.pyplot as plt
import sunpy.map
from sunpy.net import Fido, attrs as a
import astropy.units as u

## Busca e download de um frame específico

Ajuste:
- `a.Time(...)` — data/janela onde procurar frames (a busca retorna todos os frames disponíveis nesse intervalo, não baixa todos);
- `a.Wavelength(...)` — comprimento de onda AIA desejado (94, 131, 171, 193, 211, 304, 335, 1600, 1700 ou 4500 Å; 2310 Å não existe no AIA);
- `result[0][<indice>]` — qual frame, dentre os encontrados na janela, baixar de fato. O índice usado no exemplo (~22) foi escolhido observando o `print(result)` para achar o frame mais próximo do horário de interesse.

In [ ]:

result = Fido.search(
    a.Time('2011-06-05 08:00', '2011-06-05 08:30'),  # janela em torno da CME
    a.Instrument('AIA'),
    a.Wavelength(1700*u.angstrom)
)

print(result)
# Frame próximo de 08:09 UT (índice ~22)
files_cme = Fido.fetch(result[0][22])

## Plotagem simples (sem projeção WCS)

Visualização rápida em pixels, sem coordenadas heliográficas. Salva a figura em `sol_1700A_cme.png` — ajuste o nome do arquivo e o título se estiver usando outra data/comprimento de onda.

In [ ]:

mapa_cme = sunpy.map.Map(files_cme[0])

# Versão simplificada sem projeção WCS
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(mapa_cme.data, origin='lower', cmap=mapa_cme.cmap,
          vmin=0, vmax=mapa_cme.data.max() * 0.3)
ax.set_title(f'Sol AIA 1700 Å — {mapa_cme.date}')
ax.set_xlabel('Pixel X')
ax.set_ylabel('Pixel Y')
plt.colorbar(ax.images[0], ax=ax, label='Intensidade')
plt.tight_layout()
plt.savefig('sol_1700A_cme.png', dpi=300, bbox_inches='tight')
plt.show()

## Conferência com projeção WCS (`Map.peek()`)

Alternativa rápida de visualização usando o método nativo do SunPy `Map.peek()`, que já plota com eixos em coordenadas heliográficas/WCS (mais completo que a plotagem em pixels acima, porém sem controle fino sobre a figura).

In [ ]:

mapa_cme = sunpy.map.Map(files_cme[0])
mapa_cme.peek()